# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [ ]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [ ]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [ ]:
# TODO: sesuaikan nama kolom dengan dataset Anda
df_missing_handled = df.with_columns([
    # Contoh: isi kolom numerik dengan median
    # pl.col('kolom_numerik').fill_null(pl.col('kolom_numerik').median()),

    # Contoh: isi kolom kategorik dengan nilai default
    # pl.col('kolom_kategorik').fill_null('tidak diketahui'),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [ ]:
# TODO: ganti 'kolom_numerik' dengan kolom numerik utama pada dataset Anda
KOLOM_NUMERIK = 'kolom_numerik'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Windsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [ ]:
# TODO: ganti 'id_unik' dengan kolom kunci unik pada dataset Anda
KOLOM_KUNCI = 'id_unik'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [ ]:
connection.register('dataset_bersih', df_dedup)

# TODO: sesuaikan aturan validasi dengan kolom dan batas nilai pada dataset Anda
validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()
validasi

## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.